# 01 — K-Means Clustering on the Iris Dataset

This notebook explores unsupervised clustering using **features only**. Iris species labels are not used to fit K-Means; they are reserved for an optional post-clustering comparison.

## Learning objectives
- Scale numerical features for distance-based clustering.
- Fit K-Means and inspect cluster assignments and centroids.
- Use inertia and the elbow method to study candidate values of K.
- Use silhouette scores as a complementary cluster-quality measure.
- Visualize clusters in a two-dimensional PCA projection.

Official reference: [Scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html#k-means).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Load and inspect the features
We load all four numeric measurements. Species labels are stored separately and never passed to K-Means during fitting.

In [ ]:
iris = load_iris()
X = iris.data
species_labels = iris.target  # reserved for post-hoc comparison only
feature_names = iris.feature_names

iris_frame = pd.DataFrame(X, columns=feature_names)
display(iris_frame.head())
print(f'Samples: {X.shape[0]} | Features: {X.shape[1]}')
print('Missing values:', int(iris_frame.isna().sum().sum()))

## 2. Scale the features
K-Means relies on Euclidean distances. Standardization reduces the chance that a feature with a larger numeric range dominates distance calculations.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled feature means:', np.round(X_scaled.mean(axis=0), 3))
print('Scaled feature standard deviations:', np.round(X_scaled.std(axis=0), 3))

## 3. Fit K-Means
`n_clusters` sets K. `n_init=10` tries multiple centroid initializations, and `random_state` makes the setup reproducible. Cluster IDs are arbitrary numbers, not species labels.

API: [sklearn.cluster.KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
cluster_labels = kmeans.fit_predict(X_scaled)

print('Inertia:', round(kmeans.inertia_, 3))
print('Cluster counts:')
print(pd.Series(cluster_labels).value_counts().sort_index())

## 4. Elbow method using inertia
Inertia is the sum of squared distances from samples to their assigned centroid. It does not increase as K increases, so do not simply choose the largest K. Look for a bend where additional clusters bring diminishing returns.

In [ ]:
candidate_k = range(1, 11)
inertias = []

for k in candidate_k:
    candidate_model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    candidate_model.fit(X_scaled)
    inertias.append(candidate_model.inertia_)

plt.figure(figsize=(8, 4.5))
plt.plot(list(candidate_k), inertias, marker='o')
plt.xticks(list(candidate_k))
plt.xlabel('Number of clusters (K)')
plt.ylabel('Inertia (within-cluster sum of squares)')
plt.title('Elbow Method for K-Means on Iris')
plt.grid(alpha=0.25)
plt.show()

## 5. Silhouette score for candidate K values
The silhouette score ranges from -1 to +1. Larger values often suggest better separation, but the metric favors some cluster geometries more than others. Use it with plots and context instead of treating it as absolute proof.
API: [silhouette_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).

In [ ]:
silhouette_results = []
for k in range(2, 11):
    candidate_model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = candidate_model.fit_predict(X_scaled)
    silhouette_results.append({'k': k, 'inertia': candidate_model.inertia_, 'silhouette_score': silhouette_score(X_scaled, labels)})

silhouette_frame = pd.DataFrame(silhouette_results)
display(silhouette_frame.round(3))

plt.figure(figsize=(8, 4.5))
plt.plot(silhouette_frame['k'], silhouette_frame['silhouette_score'], marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Silhouette score')
plt.title('Silhouette Score by K')
plt.grid(alpha=0.25)
plt.show()

## 6. Visualize clusters in two dimensions
PCA reduces the scaled four-dimensional measurements to two components for visualization. K-Means is still trained on all four scaled features.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', s=45, alpha=0.85)
plt.xlabel('PCA component 1')
plt.ylabel('PCA component 2')
plt.title('K-Means Clusters on Iris (PCA projection)')
plt.legend(*scatter.legend_elements(), title='Cluster')
plt.grid(alpha=0.15)
plt.show()
print('Explained variance ratio:', np.round(pca.explained_variance_ratio_, 3))

## 7. Optional post-hoc comparison with Iris species
The Iris dataset includes known species, so we can inspect how clusters overlap with species after fitting. Species labels were not used to train K-Means. Cluster IDs are arbitrary and do not directly correspond to species numbers.

In [ ]:
comparison = pd.crosstab(
    pd.Series(cluster_labels, name='K-Means cluster'),
    pd.Series([iris.target_names[label] for label in species_labels], name='Known species')
)
display(comparison)

## Key takeaways
- K-Means needs K in advance and aims to reduce inertia.
- The elbow plot and silhouette score answer different questions; inspect both.
- Feature scaling matters when clusters use distances.
- PCA is used for visualization; clustering used all original measurements.
- Cluster numbers are arbitrary and do not inherently name groups.